In [1]:
# %%
# IMPORT LIBRARIES
import ee
import geemap
import pandas as pd

# %%
# INITIALIZE EARTH ENGINE
ee.Initialize(project='perfect-impulse-474508-e2')

# %%
# LOAD CSV DATA
csv_path = '/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_processed.csv'
df = pd.read_csv(csv_path)

# CLEAN UP DATA: REMOVE ROWS WITH MISSING COORDINATES OR EPC
df_clean = df.dropna(subset=['LATITUDE', 'LONGITUDE', 'CURRENT_ENERGY_RATING'])

# %%
# CREATE EE FEATURES FROM CSV
def make_feature(row):
    geom = ee.Geometry.Point([row['LONGITUDE'], row['LATITUDE']])
    props = {'CURRENT_ENERGY_RATING': row['CURRENT_ENERGY_RATING']}
    return ee.Feature(geom, props)

features = [make_feature(row) for idx, row in df_clean.iterrows()]
houses_fc = ee.FeatureCollection(features)

# %%
# MAP EPC LETTERS TO NUMERIC VALUES (A=1, ..., G=7)
epc_order = ['A','B','C','D','E','F','G']

def epc_numeric(feature):
    epc = ee.String(feature.get('CURRENT_ENERGY_RATING'))
    # Map letter to number using index in list +1
    num = ee.List(epc_order).indexOf(epc).add(1)
    return feature.set('EPC_num', num)

houses_fc_num = houses_fc.map(epc_numeric)

# %%
# BUFFER POINTS FOR VISIBILITY
houses_fc_buffered = houses_fc_num.map(lambda f: f.buffer(7))  # 20m radius

# %%
# VISUALIZATION PARAMETERS
palette = ['#006400', '#00A000', '#78B300', '#FFFF00', '#FFA500', '#FF4500', '#FF0000']
vis_params = {'min': 1, 'max': 7, 'palette': palette}

# Get number of houses included in the map
num_houses = houses_fc_num.size().getInfo()
print(f"Number of houses included: {num_houses}")

# CREATE MAP CENTERED ON DATA
center_lat = df_clean['LATITUDE'].mean()
center_lon = df_clean['LONGITUDE'].mean()
m = geemap.Map(center=[center_lat, center_lon], zoom=14)
m.add_basemap('SATELLITE')

# PAINT FEATURES AS IMAGE FOR VISUALIZATION
points_image = ee.Image().paint(houses_fc_buffered, color='EPC_num', width=3)
m.add_ee_layer(points_image, vis_params, 'Houses by EPC')

m.add_layer_control()
m

Number of houses included: 117


Map(center=[np.float64(56.05654801666498), np.float64(-4.225281702153846)], controls=(WidgetControl(options=['…